In [1]:
import kagglehub

path = kagglehub.dataset_download("suraj520/multi-task-learning")
import glob
print("Path to dataset files:", path)
print(glob.glob(f"{path}/*.csv"))
import pandas
data = pandas.read_csv(f"{path}/data.csv")
print(data)
X = data["tweet"]
y = data["sentiment"]

/home/ceres/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/ceres/.cache/kagglehub/datasets/suraj520/multi-task-learning/versions/1
['/home/ceres/.cache/kagglehub/datasets/suraj520/multi-task-learning/versions/1/data.csv']
                                                  tweet language sentiment
0     Lionel Messi, que ha estado vinculado con un t...       es   3 stars
1     This is a guest post by The Joy of Truth. To r...       en   4 stars
2     Nous sommes tous conscients de la popularité d...       fr   5 stars
3     El baño en el sistema de metro de la ciudad de...       es   4 stars
4     "Ich habe dies seit über 20 Jahren getan und i...       de   5 stars
...                                                 ...      ...       ...
4912  \nA former CIA officer and CIA director has pl...       en    1 star
4913  Karen M. Felt, Ph.D. La ricerca è stata condot...       it   4 stars
4914  Mit all der Aufmerksamkeit, die dem Thema Abtr...       de   2 stars
4915  L'élément le plus important dans le processus ...   

In [2]:
#Et maintenant on ré-utilise la même chaîne de traitement !
from tools_pretraitement import *
import re
from pathlib import Path

import numpy 
import pandas

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score

/home/ceres/.local/lib/python3.10/site-packages/stopwordsiso/_core.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
PREPROCESSORS = {
    "DON": don,
    "LOW": lower,
    "URLrem": remove_urls,
    "URLrep": replace_urls,
    "PUN": remove_punct,
    "RSW": remove_stopwords,
    "LOW+URLrem": compose(lower, remove_urls),
    "LOW+URLrep": compose(lower, replace_urls),
    "LOW+PUN": compose(lower, remove_punct),
    "LOW+URLrem+PUN": compose(lower, remove_urls, remove_punct),
    "LOW+URLrep+PUN": compose(lower, remove_urls, remove_punct),
    "LOW+URLrem+PUN+RSW": compose(lower, remove_urls, remove_punct, remove_stopwords),
    "LOW+URLrep+PUN+RSW": compose(lower, replace_urls, remove_punct, remove_stopwords),

}
toto = "I am travelling to Nancy for an NLP course at IDMC :https://idmc.univ-lorraine.fr/" 
for prep_name, prep in PREPROCESSORS.items():
    print(f"Prep: {prep_name}")
    print(prep(toto))

Prep: DON
I am travelling to Nancy for an NLP course at IDMC :https://idmc.univ-lorraine.fr/
Prep: LOW
i am travelling to nancy for an nlp course at idmc :https://idmc.univ-lorraine.fr/
Prep: URLrem
I am travelling to Nancy for an NLP course at IDMC :
Prep: URLrep
I am travelling to Nancy for an NLP course at IDMC : <URL> 
Prep: PUN
I am travelling to Nancy for an NLP course at IDMC  https   idmc univ lorraine fr 
Prep: RSW
I travelling Nancy NLP IDMC : https : / / idmc . univ - lorraine . /
Prep: LOW+URLrem
i am travelling to nancy for an nlp course at idmc :
Prep: LOW+URLrep
i am travelling to nancy for an nlp course at idmc : <URL>
Prep: LOW+PUN
i am travelling to nancy for an nlp course at idmc https idmc univ lorraine fr
Prep: LOW+URLrem+PUN
i am travelling to nancy for an nlp course at idmc
Prep: LOW+URLrep+PUN
i am travelling to nancy for an nlp course at idmc
Prep: LOW+URLrem+PUN+RSW
travelling nancy nlp idmc
Prep: LOW+URLrep+PUN+RSW
travelling nancy nlp idmc URL


In [4]:
#Evaluation part
from tools_eval import *

macro_f1 = make_scorer(f1_score, average="macro")
SCORING = {"acc": "accuracy", "macro_f1": macro_f1}
def mean_scores(scores):
    return {k.replace("test_", ""): float(numpy.mean(v))
            for k, v in scores.items() if k.startswith("test_")}

def evaluate(X, y, preprocess, vectorizer):
    Xp = [preprocess(t) for t in X]
    pipe = Pipeline([
        ("vect", vectorizer),
        ("clf", MODEL),
    ])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = cross_validate(pipe, Xp, y, cv=cv, scoring=SCORING, n_jobs=-1)
    return mean_scores(scores)


In [6]:
#Partie apprentissage
SEED = 42
MODEL = LogisticRegression(max_iter=2000, random_state=SEED)
OUTFILE = Path("results_autorship-attribution.csv")

import os
os.environ["PYTHONWARNINGS"] = "ignore:pkg_resources is deprecated as an API:UserWarning"


VECTORIZERS = {
    #"count_word_1-1": CountVectorizer(analyzer="word", ngram_range=(1, 1), lowercase=False),
    #"count_word_1-2": CountVectorizer(analyzer="word", ngram_range=(1, 2), lowercase=False),
    #"tfidf_word_1-1": TfidfVectorizer(analyzer="word", ngram_range=(1, 1), lowercase=False),
    #"tfidf_word_1-2": TfidfVectorizer(analyzer="word", ngram_range=(1, 2), lowercase=False),
    "count_char_2-4": CountVectorizer(analyzer="char", ngram_range=(2, 4), lowercase=False),
    "count_char_2-4": CountVectorizer(analyzer="char", ngram_range=(2, 4), lowercase=False, max_features=10000),
    "count_char_4-6": CountVectorizer(analyzer="char", ngram_range=(4, 6), lowercase=False, max_features=10000),
    #"count_charwb_3-5": CountVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=False),
    #"tfidf_char_3-5": TfidfVectorizer(analyzer="char", ngram_range=(3, 5), lowercase=False),
    #"tfidf_charwb_3-5": TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=False),
}

rows = []
for prep_name, prep in PREPROCESSORS.items():
    for vec_name, vec in VECTORIZERS.items():
        print(f"Prep: {prep_name:12s} | Vec: {vec_name}")
        res = evaluate(X, y, prep, vec)
        print(res)
        rows.append({
            "preprocessing": prep_name,
            "vectorizer": vec_name,
            **res
        })

Prep: DON          | Vec: count_char_2-4
{'acc': 0.44132922280392695, 'macro_f1': 0.39971597724557484}
Prep: DON          | Vec: count_char_4-6
{'acc': 0.4370561744783267, 'macro_f1': 0.3925675808231288}
Prep: LOW          | Vec: count_char_2-4
{'acc': 0.4498703570453813, 'macro_f1': 0.4042664722737491}
Prep: LOW          | Vec: count_char_4-6
{'acc': 0.4458013878205923, 'macro_f1': 0.40024401779060587}
Prep: URLrem       | Vec: count_char_2-4
{'acc': 0.44275198703156915, 'macro_f1': 0.40071986472208465}
Prep: URLrem       | Vec: count_char_4-6
{'acc': 0.4376646899734511, 'macro_f1': 0.3925359038883839}
Prep: URLrep       | Vec: count_char_2-4
{'acc': 0.4419385653673424, 'macro_f1': 0.40033449188074777}
Prep: URLrep       | Vec: count_char_4-6
{'acc': 0.4340051195527215, 'macro_f1': 0.39012055124861955}
Prep: PUN          | Vec: count_char_2-4
{'acc': 0.433596754584026, 'macro_f1': 0.38875244558806615}
Prep: PUN          | Vec: count_char_4-6
{'acc': 0.42973186446004846, 'macro_f1': 0.

In [ ]:
df = pandas.DataFrame(rows).sort_values("macro_f1", ascending=False)
print(df)
OUTFILE = "Classif_sentiment_multiL_feats.csv"
df.to_csv(OUTFILE, index=False)
print(f"Saved to: {OUTFILE}")
